In [9]:

from MEA1D.MEA1D_conf import *

In [10]:
class anode:
    
    def __init__(self):
        self.U = 1.15
        self.yb_jp = 0
        self.yb_jlambda = 0
        self.yb_T = 0
        
    def dydx(self,x,y):
        phi_e, j_e, phi_p, j_p, T, j_T, lambda_, j_lambda, x_H2O, j_H2O, x_H2, j_H2, x_O2, j_O2, s, j_s = [y[i_var, :]for i_var in range(16)]
        dphi_e, dj_e, dphi_p, dj_p, dT, dj_T, dlambda, dj_lambda, dx_H2O, dj_H2O, dx_H2, dj_H2, dx_O2, dj_O2, ds, dj_s = [np.zeros(y.shape[1]) for _ in range(16)]
        # AGDL
        C = P_A / (R * T[:11])  # total interstitial gas concentration
        dphi_e[:11] = -j_e[:11] / sigma_e_GDL  # electron flux: j_e = -sigma_e*grad(phi_e)
        dT[:11] = -j_T[:11] / k_GDL  # heat flux: j_T = -k*grad(T)
        dx_H2O[:11] = -j_H2O[:11] / (
                C * D_H2O_A(eps_p_GDL, tau_GDL, s[:11], T[:11]))  # water vapor flux: j_H2O = -C*D_H2O*grad(x_H2O)
        dx_H2[:11] = -j_H2[:11] / (C * D_H2(eps_p_GDL, tau_GDL, s[:11], T[:11]))  # hydrogen flux: j_H2 = -C*D_H2*grad(x_H2)
        dj_T[:11] = -j_e[:11] * dphi_e[:11]  # conservation of heat: div(j_T) = S_T
        # ACL
        C = P_A / (R * T[11:22])  # total interstitial gas concentration
        x_sat = P_sat(T[11:22]) / P_A  # saturation water vapor mole fraction
        lambda_eq = sorption(x_H2O[11:22] / x_sat)  # equilibrium water content of ionomer
        S_ad = k_ad(lambda_[11:22], lambda_eq, T[11:22]) / (L[2] * V_m) * (
                lambda_eq - lambda_[11:22])  # absorption/desorption reaction rate
        eta = phi_e[11:22] - phi_p[11:22] + T[11:22] * DeltaS_HOR / (2 * F) + R * T[11:22] / (2 * F) * np.log(
            x_H2[11:22] * P_A / P_ref)  # overpotential
        i = BV(i_0_HOR(T[11:22]), a_ACL, T[11:22], beta_HOR, eta)  # electrochemical reaction rate
        S_F = i / (2 * F)  # Faraday's law
        dphi_e[11:22] = -j_e[11:22] / sigma_e_CL  # electron flux: j_e = -sigma_e*grad(phi_e)
        dphi_p[11:22] = -j_p[11:22] / sigma_p(eps_i_CL, lambda_[11:22],T[11:22])  # proton flux: j_p = -sigma_p*grad(phi_p)
        dT[11:22] = -j_T[11:22] / k_CL  # heat flux: j_T = -k*grad(T)
        dlambda[11:22] = (-j_lambda[11:22] + xi(lambda_[11:22]) / F * j_p[11:22]) * V_m / D_lambda(eps_i_CL,lambda_[11:22],T[11:22])  # dissolved water flux: j_lambda = -D_lambda/V_m*grad(lambda)+xi/F*j_p
        dx_H2O[11:22] = -j_H2O[11:22] / (C * D_H2O_A(eps_p_CL, tau_CL, s[11:22], T[11:22]))  # water vapor flux: j_H2O = -C*D_H2O*grad(x_H2O)
        dx_H2[11:22] = -j_H2[11:22] / (C * D_H2(eps_p_CL, tau_CL, s[11:22], T[11:22]))  # hydrogen flux: j_H2 = -C*D_H2*grad(x_H2)
        dj_e[11:22] = -i  # conservation of electrons: div(j_e) = S_e
        dj_p[11:22] = i  # conservation of protons: div(j_p) = S_p
        dj_T[11:22] = -j_e[11:22] * dphi_e[11:22] - j_p[11:22] * dphi_p[11:22] + i * eta - S_F * T[11:22] * DeltaS_HOR + H_ad * S_ad  # conservation of heat: div(j_T) = S_T
        dj_lambda[11:22] = S_ad  # conservation of dissolved water: div(j_lambda) = S_lambda
        dj_H2O[11:22] = -S_ad  # conservation of water vapor: div(j_H2O) = S_H2O
        dj_H2[11:22] = -S_F  # conservation of hydrogen: div(j_H2) = S_H2
        
        self.yb_jp = 1e-5*dj_p[-1]*11
        self.yb_jlambda = 1e-5*dj_lambda[-1]*11
        self.yb_T = 1e-5*dT[-1]*11
        
        return [dphi_e, dj_e,
                dphi_p, dj_p,
                dT, dj_T,
                dlambda, dj_lambda,
                dx_H2O, dj_H2O,
                dx_H2, dj_H2,
                dx_O2, dj_O2,
                ds, dj_s]
    
    def y_init(self, x):
        
        y = np.zeros((16, len(x)))
        for n in range(2):
            for i in range(11):
                if n == 0:
                    y[:, n * 11 + i] = self.y_init_agdl(x)
                    if i > 0:
                        dy = 1.6e-5 * np.array(self.dydx(x, y))[:,n * 11 + i-1]
                        y[:, n * 11 + i] = y[:, n * 11 + i] + dy
                else:
                    y[:, n * 11 + i] = self.y_init_acl(x)
                    if i > 0:
                        dy = 1.0e-6 * np.array(self.dydx(x, y))[:,n * 11 + i-1]
                        y[:, n * 11 + i] = y[:, n * 11 + i] + dy
        return y
    
    def y_init_agdl(self, x):
        T = (T_C + T_A) / 2
        x_H2O = x_H2O_A
        x_H2 = x_H2_A
        return np.array([0, 0, 0, 0, T, 0, 0, 0, x_H2O, 0, x_H2, 0, 0, 0, 0, 0])

    def y_init_acl(self,x):
        lambda_ = sorption(1)
        T = (T_C + T_A) / 2
        x_H2O = x_H2O_A
        x_H2 = x_H2_A
        return np.array([0, 0, 0, 0, T, 0, lambda_, 0, x_H2O, 0, x_H2, 0, 0, 0, 0, 0])
    
    
    def bcfun(self, ya, yb):
        res = np.zeros(len(ya))
        # ELECTRONS
        res[0] = ya[1]
        res[1] = yb[1]

        # PROTONS
        res[2] = ya[2]
        res[3] = yb[2] - self.yb_jp

        # TEMPERATURE
        res[4] = ya[4] - T_A
        res[5] = yb[4] - T_A - self.yb_T

        # DISSOLVED WATER
        res[6] = ya[7]
        res[7] = ya[7] - sorption(1) - self.yb_jlambda

        # WATER VAPOR
        res[8] = ya[8] - x_H2O_A
        res[9] = yb[8]

        # HYDROGEN
        res[10] = ya[10] - x_H2_A
        res[11] = yb[10]

        # OXYGEN
        res[12] = ya[12]
        res[13] = yb[12]

        # LIQUID WATER
        res[14] = ya[14]
        res[15] = yb[14]

        return np.array(res)

In [11]:
model = anode()

In [13]:
model.y_init_agdl(np.linspace(0,1.6e-4,11))

array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       3.43150000e+02, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       1.87017673e-01, 0.00000000e+00, 8.12982327e-01, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00])